# Comparación de distribuciones: MPS frente a Gibbs y Rényi

Compara, a una temperatura e índice `IDX` seleccionados, el histograma de energías locales del estado ARViT+MPS entrenado con las distribuciones Gibbs y Rényi-2 construidas desde la densidad de estados del Ising transversal. El estado y sus parámetros se cargan del formato MPS (`.msgpack`) y el muestreo se hace con los helpers JAX del proyecto, sin NetKet.

Para N grande no se construye una matriz densa de dimensión `2**N`: la DOS se calcula por convolución de los dos sectores de Jordan–Wigner. Es la solución exacta de fermiones libres del modelo, representada en una rejilla de energía finita; no es una diagonalización densa numérica.

In [ ]:
import json
import sys
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src_renyi').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src_renyi').is_dir():
    raise FileNotFoundError('No se encuentra la raíz del proyecto (src_renyi).')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
jax.config.update('jax_enable_x64', True)

from src_renyi.mps_ancilla.ladder import build_model, load_params
from src_renyi.mps_ancilla.models import mps_overlap_aligned
from src_renyi.mps_ancilla.training import make_sampler
from scripts import ff_local_obs as ff   # MRE/Gibbs exactos a N finito (fermiones libres)

# Selectores editables
N = 15
IDX = 15
CHI = 8
N_SAMPLES = 4096
ENERGY_CHUNK = 128
SEED = 123
DOS_STEP = 0.005
DOS_BINS = 250

DATA_DIR = PROJECT_ROOT / 'data' / '0' / f'mps_N{N}'
RESULTS_FILE = DATA_DIR / f'results_mps_N{N}_vs_T.json'
PARAMS_FILE = DATA_DIR / 'params' / f'params_{IDX:04d}_chi{CHI}.msgpack'
if not RESULTS_FILE.is_file():
    raise FileNotFoundError(f'No existe el archivo de resultados: {RESULTS_FILE}')
if not PARAMS_FILE.is_file():
    raise FileNotFoundError(f'No existen los parámetros entrenados: {PARAMS_FILE}')

with RESULTS_FILE.open() as f:
    results = json.load(f)
if IDX < 0 or IDX >= len(results.get('T', [])):
    raise IndexError(f'IDX={IDX} fuera del barrido de temperaturas guardado.')
chi_results = results.get('by_chi', {}).get(str(CHI))
if chi_results is None:
    raise KeyError(f'chi={CHI} no aparece en los resultados guardados.')
if IDX >= len(chi_results.get('energy', [])) or chi_results['energy'][IDX] is None:
    raise IndexError(f'No hay resultados entrenados para IDX={IDX}, chi={CHI}.')

T = float(results['T'][IDX])
if not np.isfinite(T) or T <= 0:
    raise ValueError(f'La temperatura debe ser positiva para esta comparación; T={T}.')
saved_energy_per_site = float(chi_results['energy'][IDX]) / N
saved_hx = chi_results['h_x'][IDX]
saved_j = chi_results['J_ZZ'][IDX]
if saved_hx is None or saved_j is None:
    raise ValueError('Los resultados no incluyen h_x y J_ZZ para esta temperatura.')
h_x, J_ZZ = float(saved_hx), float(saved_j)
if not np.isfinite(h_x) or not np.isfinite(J_ZZ):
    raise ValueError('Los parámetros h_x y J_ZZ guardados no son finitos.')

print(f'N={N}, IDX={IDX}, chi={CHI}, T={T:g}, h_x={h_x:g}, J_ZZ={J_ZZ:g}')
print(f'Energía guardada del entrenamiento: {saved_energy_per_site:.6f} por sitio')

## DOS y distribuciones analíticas

La convolución conserva los sectores de paridad NS/R del TFIM periódico. La rejilla `DOS_STEP` discretiza la DOS para poder trabajar con N=25 sin enumerar los `2**N` estados.

In [ ]:
def exact_tfim_dos(N, J, h, dE_total):
    if N < 2 or J <= 0 or h <= J or dE_total <= 0:
        raise ValueError('Se requiere N>=2, J>0, h>J y DOS_STEP>0 para esta DOS TFIM.')

    def sector_dos(k, required_parity):
        eps = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k))
        n_bins = int(eps.sum() / dE_total) + 3
        counts = np.zeros((2, n_bins), dtype=float)
        counts[0, 0] = 1.0
        for energy in eps:
            shift = energy / dE_total
            lower = int(shift)
            fraction = shift - lower
            updated = counts.copy()
            for parity in (0, 1):
                updated[1 - parity, lower:] += (1 - fraction) * counts[parity, :n_bins - lower]
                updated[1 - parity, lower + 1:] += fraction * counts[parity, :n_bins - lower - 1]
            counts = updated
        energies = -0.5 * eps.sum() + dE_total * np.arange(n_bins)
        return energies, counts[required_parity]

    k_ns = 2 * np.pi * (np.arange(N) + 0.5) / N
    k_r = 2 * np.pi * np.arange(N) / N
    energies_ns, weights_ns = sector_dos(k_ns, 0)
    energies_r, weights_r = sector_dos(k_r, 1)
    energies = np.concatenate((energies_ns, energies_r))
    weights = np.concatenate((weights_ns, weights_r))
    grid = np.arange(energies.min(), energies.max() + 1.5 * dE_total, dE_total)
    edges = np.append(grid - dE_total / 2, grid[-1] + dE_total / 2)
    dos_counts, _ = np.histogram(energies, bins=edges, weights=weights)
    e_per_site = grid / N
    de_per_site = dE_total / N
    dos = dos_counts / (dos_counts.sum() * de_per_site)
    return e_per_site, dos, de_per_site


def normalize_density(weights, de):
    norm = np.sum(weights) * de
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError('No se pudo normalizar la distribución de energía.')
    return weights / norm


E_DOS, DOS, dE = exact_tfim_dos(N, abs(J_ZZ), abs(h_x), DOS_STEP)
beta = 1.0 / T
log_gibbs = np.log(DOS + 1e-300) - beta * N * E_DOS
weights_gibbs = np.exp(log_gibbs - log_gibbs.max())
p_gibbs = normalize_density(weights_gibbs, dE)

E_bar = np.sum(E_DOS * p_gibbs) * dE
for _ in range(1000):
    renyi_weight = DOS * np.maximum(1.0 - 0.5 * beta * N * (E_DOS - E_bar), 0.0)
    p_renyi = normalize_density(renyi_weight, dE)
    E_next = np.sum(E_DOS * p_renyi) * dE
    if abs(E_next - E_bar) < 1e-10:
        E_bar = E_next
        break
    E_bar = E_next

mean_gibbs = np.sum(E_DOS * p_gibbs) * dE
mean_renyi = np.sum(E_DOS * p_renyi) * dE
print(f'<E>/N Gibbs = {mean_gibbs:.6f}; Rényi-2 = {mean_renyi:.6f}')

## Muestreo del vstate entrenado

Se muestrean configuraciones del autoregresor del sistema. Para cada configuración se calcula la energía local con las amplitudes de configuraciones volteadas y los solapes normalizados de la ancilla MPS, igual que en `mc_loss.py`.

In [ ]:
model = build_model(N, CHI, d=16, heads=4, blocks=2, mps_init=0.05)
params = load_params(str(PARAMS_FILE), model, N)
sampler = make_sampler(model, N, N_SAMPLES)
samples = np.asarray(sampler(params, jax.random.PRNGKey(SEED)))

def local_energy_per_site(sample_batch):
    batch_size = sample_batch.shape[0]
    log_a, m = model.apply({'params': params}, sample_batch)
    norm2 = jnp.real(mps_overlap_aligned(m, m, CHI))
    flips = 1.0 - 2.0 * jnp.eye(N, dtype=sample_batch.dtype)
    flipped = (sample_batch[:, None, :] * flips[None, :, :]).reshape(batch_size * N, N)
    log_a_flipped, m_flipped = model.apply({'params': params}, flipped)
    m_repeated = jnp.repeat(m, N, axis=0)
    overlap = mps_overlap_aligned(m_flipped, m_repeated, CHI)
    norm2_flipped = jnp.real(mps_overlap_aligned(m_flipped, m_flipped, CHI))
    overlap /= jnp.sqrt(norm2_flipped * jnp.repeat(norm2, N) + 1e-300)
    amplitude_ratio = jnp.exp(log_a_flipped - jnp.repeat(log_a, N))
    off_diagonal = jnp.sum((amplitude_ratio * overlap).reshape(batch_size, N), axis=1)
    diagonal = jnp.sum(sample_batch * jnp.roll(sample_batch, -1, axis=1), axis=1)
    return jnp.real(h_x * off_diagonal + J_ZZ * diagonal) / N

local_energies = []
for start in range(0, N_SAMPLES, ENERGY_CHUNK):
    batch = jnp.asarray(samples[start:start + ENERGY_CHUNK])
    local_energies.append(np.asarray(local_energy_per_site(batch)))
local_energies = np.concatenate(local_energies)
if not np.all(np.isfinite(local_energies)):
    raise FloatingPointError('El estimador de energía local produjo valores no finitos.')

mean_vstate = float(local_energies.mean())
print(f'<E>/N vstate (muestras) = {mean_vstate:.6f}')
print(f'<E>/N vstate (resultados guardados) = {saved_energy_per_site:.6f}')

In [ ]:
hist_density, hist_edges = np.histogram(local_energies, bins=DOS_BINS, density=True)
hist_centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
hist_width = hist_edges[1] - hist_edges[0]
hist_density = normalize_density(hist_density, hist_width)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(E_DOS, p_gibbs, color='tab:blue', lw=2.2, zorder=2, label=f'Gibbs, T={T:g}')
ax.plot(E_DOS, p_renyi, color='tab:red', lw=2.2, ls='--', zorder=3,
        label=f'Rényi-2 analítico, T={T:g}')
ax.bar(hist_centers, hist_density, width=hist_width, color='tab:orange',
       edgecolor='darkorange', linewidth=0.35, alpha=0.38, zorder=4,
       label=f'vstate MPS, chi={CHI} ({N_SAMPLES} muestras)')
ax.set_xlabel(r'$E/N$')
ax.set_ylabel(r'$p(E/N)$')
ax.set_title(f'Distribución de energía a T={T:g}: N={N}, chi={CHI}, IDX={IDX}')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Medias por sitio: vstate={mean_vstate:.6f}, Gibbs={mean_gibbs:.6f}, Rényi-2={mean_renyi:.6f}')

## Distribución de energía real frente a energías locales

**Ojo:** el histograma de arriba es de las *energías locales* $E_\mathrm{loc}(s)=\sum_t \rho(s,t)H(t,s)/\rho(s,s)$ con $s\sim p(s)$. Su media es $\langle H\rangle$, pero su distribución **no** es la distribución de energía del estado, $q(E_n)=\langle n|\rho|n\rangle$, que es la que hay que comparar con Gibbs y Rényi-2. Es mucho más estrecha: a N=8, $\mathrm{Var}(E_\mathrm{loc})=1.17$ frente a $\mathrm{Var}(H)=9.14$; a N=25 (X5), 3.4 frente a 17.

A N grande no se puede obtener $q(E)$ entera, pero sí su varianza exacta por MC (como en `scripts/var_h.py`):
$\mathrm{Tr}(\rho H^2)=\mathbb{E}_{s\sim p}[E_{2,\mathrm{loc}}(s)]$, con
$E_{2,\mathrm{loc}}(s)=D(s)^2+N h_x^2+h_x\sum_i (D(s)+D(s^i))\,r(s\to s^i)+2h_x^2\sum_{i<j} r(s\to s^{ij})$.

Abajo se dibuja una **gaussiana con la media y la $\mathrm{Var}(H)$ del modelo**. Es solo una referencia de anchura, porque iguala los dos primeros momentos y no la forma: la cola por encima del corte del MRE no se ve así. Para la forma exacta a N ≤ 12, usa `scripts/s2_deficit_split.py` y `scripts/plot_qE_hist.py`; los plots están en `plots/histograms/`.

In [ ]:
VARH_CHUNK = 16   # muestras por trozo en los dobles flips (N(N-1)/2 configuraciones por muestra)

_I, _J = np.triu_indices(N, 1)
_double = np.ones((len(_I), N)); _double[np.arange(len(_I)), _I] = -1.0; _double[np.arange(len(_I)), _J] = -1.0
_single = jnp.asarray(1.0 - 2.0 * np.eye(N)); _double = jnp.asarray(_double)

def _diag_energy(s):
    return J_ZZ * jnp.sum(s * jnp.roll(s, -1, axis=-1), axis=-1)

@jax.jit
def local_E_and_E2(sample_batch):
    # E_loc(s) y E2_loc(s) (energía total, no por sitio); misma razón r(s->t) que local_energy_per_site
    B = sample_batch.shape[0]
    log_a, m = model.apply({'params': params}, sample_batch)
    norm2 = jnp.real(mps_overlap_aligned(m, m, CHI))

    def ratios(masks):
        K = masks.shape[0]
        flipped = (sample_batch[:, None, :] * masks[None]).reshape(B * K, N)
        log_af, mf = model.apply({'params': params}, flipped)
        ov = mps_overlap_aligned(mf, jnp.repeat(m, K, axis=0), CHI)
        n2f = jnp.real(mps_overlap_aligned(mf, mf, CHI))
        ov = ov / jnp.sqrt(n2f * jnp.repeat(norm2, K) + 1e-300)
        return (jnp.exp(log_af - jnp.repeat(log_a, K)) * ov).reshape(B, K), flipped.reshape(B, K, N)

    D = _diag_energy(sample_batch)
    r1, s1 = ratios(_single)
    r2, _ = ratios(_double)
    E1 = jnp.real(D + h_x * jnp.sum(r1, axis=1))
    E2 = jnp.real(D**2 + N * h_x**2 + h_x * jnp.sum((D[:, None] + _diag_energy(s1)) * r1, axis=1)
                  + 2 * h_x**2 * jnp.sum(r2, axis=1))
    return E1, E2

E1_all, E2_all = [], []
for start in range(0, N_SAMPLES, VARH_CHUNK):
    e1, e2 = local_E_and_E2(jnp.asarray(samples[start:start + VARH_CHUNK]))
    E1_all.append(np.asarray(e1)); E2_all.append(np.asarray(e2))
E1_all, E2_all = np.concatenate(E1_all), np.concatenate(E2_all)

# Var(H) = <E2_loc> - <E_loc>^2 + Var(E_loc)/M (corrección de sesgo); error por bloques de 256 muestras
var_H = E2_all.mean() - E1_all.mean()**2 + E1_all.var(ddof=1) / N_SAMPLES
_blk = np.array([b2.mean() - b1.mean()**2 + b1.var(ddof=1) / len(b1)
                 for b1, b2 in zip(np.split(E1_all, N_SAMPLES // 256), np.split(E2_all, N_SAMPLES // 256))])
var_H_err = _blk.std(ddof=1) / np.sqrt(len(_blk))
var_Eloc = E1_all.var(ddof=1)

def _var_per_site(p):
    m = np.sum(E_DOS * p) * dE
    return np.sum((E_DOS - m)**2 * p) * dE

print(f'Var(H) del modelo        = {var_H:.3f} +- {var_H_err:.3f}   (std/N = {np.sqrt(var_H)/N:.4f})')
print(f'Var(E_loc) (histograma)  = {var_Eloc:.3f}             (std/N = {np.sqrt(var_Eloc)/N:.4f})')
print(f'Var(H) Gibbs, T={T:g}     = {_var_per_site(p_gibbs) * N**2:.3f}')
print(f'Var(H) Rényi-2, T={T:g}   = {_var_per_site(p_renyi) * N**2:.3f}')
print('(Gibbs y Rényi están a la T de entrenamiento, no a la energía del modelo: compáralos solo de forma orientativa)')

mu_site, sd_site = E1_all.mean() / N, np.sqrt(var_H) / N
gauss = np.exp(-0.5 * ((E_DOS - mu_site) / sd_site)**2) / (np.sqrt(2 * np.pi) * sd_site)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(E_DOS, p_gibbs, color='tab:blue', lw=2.2, label=f'Gibbs, T={T:g}')
ax.plot(E_DOS, p_renyi, color='tab:red', lw=2.2, ls='--', label=f'Rényi-2 analítico, T={T:g}')
ax.bar(hist_centers, hist_density, width=hist_width, color='0.6', alpha=0.35,
       label='energías locales $E_{loc}$ (NO es $q(E)$)')
ax.plot(E_DOS, gauss, color='tab:orange', lw=2.2, ls=':',
        label=r'gaussiana con media y Var(H) del modelo')
ax.set_xlim(mu_site - 5 * sd_site, mu_site + 5 * sd_site)
ax.set_xlabel(r'$E/N$'); ax.set_ylabel(r'$p(E/N)$')
ax.set_title(f'Anchura real de la distribución de energía: N={N}, chi={CHI}, T={T:g}')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Observables locales frente a energía por sitio

Configura `N_LIST` y `CHI_LIST` como listas emparejadas: `CHI_LIST[i]` selecciona el chi usado para `N_LIST[i]`. Para cada temperatura con checkpoint disponible se estiman con el vstate MPS $\langle\sigma^x\rangle$, $\langle\sigma^z_i\sigma^z_{i+1}\rangle$ y $\langle\sigma^x_i\sigma^x_{i+1}\rangle$. Se comparan con la solución analítica de Gibbs en el límite termodinámico del TFIM, parametrizada por la energía por sitio.

## Observables locales frente a T a N fijo: vstate frente a Rényi-2 exacto

Para el `N` y el `CHI` del principio del notebook se recorren todas las temperaturas con checkpoint y se estiman con el vstate MPS $\langle H\rangle/N$, $\langle\sigma^x\rangle$, $\langle\sigma^z_i\sigma^z_{i+1}\rangle$ y $\langle\sigma^x_i\sigma^x_{i+1}\rangle$ (con `make_mps_local_observables`, definida arriba). La referencia es el estado Rényi-2 (MRE) **exacto al mismo N**, $\rho\propto\sum_n (c-E_n)_+|n\rangle\langle n|$ con $c-\langle E\rangle=2T$, calculado con fermiones libres (`scripts/ff_local_obs.py`: estados de Fock físicos de los dos sectores de paridad y Wick por estado; validado con ED a N≤11). Se muestrean `FIXN_FF_SAMPLES` estados de Fock con peso $(c-E_n)_+$, de ahí las barras de error de la curva Rényi. Abajo, la diferencia vstate − Rényi.

In [ ]:
# Observables locales frente a T a N fijo (vstate vs Rényi-2 exacto a ese N)

FIXN_SAMPLES = 1024        # muestras del vstate por temperatura
FIXN_CHUNK = 64
FIXN_SEED = 777
FIXN_FF_SAMPLES = 20000    # estados de Fock muestreados del MRE por temperatura
FIXN_T_MIN = 0.05

def make_mps_local_observables(model, N, chi):
    single_flips = np.ones((N, N), dtype=float)
    single_flips[np.arange(N), np.arange(N)] = -1.0
    pair_flips = np.ones((N, N), dtype=float)
    for site in range(N):
        pair_flips[site, site] = -1.0
        pair_flips[site, (site + 1) % N] = -1.0
    single_flips = jnp.asarray(single_flips)
    pair_flips = jnp.asarray(pair_flips)

    @jax.jit
    def estimate(params, samples):
        batch_size = samples.shape[0]
        log_a, m = model.apply({'params': params}, samples)
        norm2 = jnp.real(mps_overlap_aligned(m, m, chi))

        def flip_ratios(flip_masks):
            n_ops = flip_masks.shape[0]
            flipped = (samples[:, None, :] * flip_masks[None, :, :]).reshape(batch_size * n_ops, N)
            log_a_flipped, m_flipped = model.apply({'params': params}, flipped)
            m_repeated = jnp.repeat(m, n_ops, axis=0)
            overlap = mps_overlap_aligned(m_flipped, m_repeated, chi)
            norm2_flipped = jnp.real(mps_overlap_aligned(m_flipped, m_flipped, chi))
            overlap = overlap / jnp.sqrt(norm2_flipped * jnp.repeat(norm2, n_ops) + 1e-300)
            ratio = jnp.exp(log_a_flipped - jnp.repeat(log_a, n_ops)) * overlap
            return ratio.reshape(batch_size, n_ops)

        sx_local = jnp.real(jnp.sum(flip_ratios(single_flips), axis=1)) / N
        szz_local = jnp.sum(samples * jnp.roll(samples, -1, axis=1), axis=1) / N
        sxx_local = jnp.real(jnp.sum(flip_ratios(pair_flips), axis=1)) / N
        return sx_local, szz_local, sxx_local

    return estimate

if not np.isclose(abs(h_x), ff.Hf) or not np.isclose(abs(J_ZZ), ff.J):
    raise ValueError(f'ff_local_obs.py asume h={ff.Hf}, J={ff.J}; aqui h_x={h_x}, J_ZZ={J_ZZ}.')

# 1) vstate: todas las T con checkpoint para (N, CHI) del principio del notebook
fixn_rows = []
for idx_t, temperature in enumerate(results['T']):
    if temperature is None or not np.isfinite(temperature) or temperature < FIXN_T_MIN:
        continue
    if idx_t >= len(chi_results['energy']) or chi_results['energy'][idx_t] is None:
        continue
    pfile = DATA_DIR / 'params' / f'params_{idx_t:04d}_chi{CHI}.msgpack'
    if pfile.is_file():
        fixn_rows.append((idx_t, float(temperature), pfile))
if not fixn_rows:
    raise ValueError(f'No hay checkpoints para N={N}, chi={CHI}.')

fixn_model = build_model(N, CHI, d=16, heads=4, blocks=2, mps_init=0.05)
fixn_sampler = make_sampler(fixn_model, N, FIXN_SAMPLES)
fixn_estimate = make_mps_local_observables(fixn_model, N, CHI)
vs = {'T': [], 'energy': [], 'sx': [], 'szz': [], 'sxx': [], 'sx_err': [], 'szz_err': [], 'sxx_err': []}
t0 = time.time()
for idx_t, temperature, pfile in fixn_rows:
    p_t = load_params(str(pfile), fixn_model, N)
    smp = fixn_sampler(p_t, jax.random.PRNGKey(FIXN_SEED + idx_t))
    acc = {'sx': [], 'szz': [], 'sxx': []}
    for st in range(0, FIXN_SAMPLES, FIXN_CHUNK):
        o = fixn_estimate(p_t, smp[st:st + FIXN_CHUNK])
        for key, v in zip(('sx', 'szz', 'sxx'), o):
            acc[key].append(np.asarray(v))
    vs['T'].append(temperature)
    for key in acc:
        v = np.concatenate(acc[key])
        vs[key].append(v.mean()); vs[key + '_err'].append(v.std() / np.sqrt(len(v)))
    vs['energy'].append(h_x * vs['sx'][-1] + J_ZZ * vs['szz'][-1])
vs = {k: np.asarray(v) for k, v in vs.items()}
print(f'vstate N={N}, chi={CHI}: {len(vs["T"])} temperaturas ({time.time() - t0:.0f} s)')

# 2) Renyi-2 (MRE) exacto a ESTE N: rho = sum_n (c - E_n)_+ |n><n| sobre estados de Fock fisicos (fermiones libres)
t0 = time.time()
ff_secs, ff_E, ff_ids, ff_sid = ff.build(N)
rng_ff = np.random.default_rng(FIXN_SEED)
re = {'T': [], 'energy': [], 'sx': [], 'szz': [], 'sxx': [], 'sx_err': [], 'szz_err': [], 'sxx_err': []}
for temperature in vs['T']:
    w, _ = ff.weights('mre', ff_E, T=temperature)
    o = ff.ensemble_obs(ff_secs, ff_E, ff_ids, ff_sid, w, N, 1, FIXN_FF_SAMPLES, rng_ff)
    re['T'].append(temperature); re['energy'].append(o['E_per_site'])
    for key, src in (('sx', 'sx'), ('szz', 'szz1'), ('sxx', 'sxx')):
        re[key].append(o[src]); re[key + '_err'].append(o[src + '_err'])
re = {k: np.asarray(v) for k, v in re.items()}
print(f'Renyi-2 exacto N={N}: {len(re["T"])} temperaturas ({time.time() - t0:.0f} s)')

fig, axes = plt.subplots(2, 4, figsize=(18, 7.5), sharex=True, gridspec_kw={'height_ratios': [2.2, 1]})
specs = [('energy', r'$\langle H\rangle/N$'), ('sx', r'$\langle\sigma^x\rangle$'),
         ('szz', r'$\langle\sigma^z_i\sigma^z_{i+1}\rangle$'), ('sxx', r'$\langle\sigma^x_i\sigma^x_{i+1}\rangle$')]
for col, (key, ylabel) in enumerate(specs):
    ax, axd = axes[0, col], axes[1, col]
    e_vs = vs.get(key + '_err'); e_re = re.get(key + '_err')
    ax.errorbar(re['T'], re[key], yerr=e_re, color='tab:red', lw=1.8, ls='--', label=f'Rényi-2 exacto, N={N}')
    ax.errorbar(vs['T'], vs[key], yerr=e_vs, fmt='o', ms=4, color='tab:orange', label=f'vstate MPS, chi={CHI}')
    ax.set_ylabel(ylabel); ax.grid(alpha=0.3)
    diff = vs[key] - re[key]
    err = None if e_vs is None else np.sqrt(e_vs**2 + e_re**2)
    axd.errorbar(vs['T'], diff, yerr=err, fmt='o', ms=3, color='tab:purple')
    axd.axhline(0, color='k', lw=0.8); axd.set_ylabel('vstate − Rényi'); axd.set_xlabel('T'); axd.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
fig.suptitle(f'Observables locales frente a T, N={N}: vstate MPS (chi={CHI}) vs Rényi-2 exacto a N={N}')
plt.tight_layout()
plt.show()

### Límite termodinámico (Gibbs, $N\to\infty$) frente a energía por sitio

In [ ]:
N_LIST = [15, 25]
CHI_LIST = [8, 4]
OBS_SAMPLES = 512
OBS_CHUNK = 64
OBS_SEED = 321
OBS_T_MIN = 0.05
MODEL_EMBEDDING, MODEL_HEADS, MODEL_BLOCKS = 16, 4, 2
GIBBS_K_POINTS = 4000

if len(N_LIST) != len(CHI_LIST) or not N_LIST:
    raise ValueError('N_LIST y CHI_LIST deben ser listas no vacías de igual longitud.')
if len(set(N_LIST)) != len(N_LIST):
    raise ValueError('N_LIST no debe contener valores repetidos.')
if OBS_SAMPLES <= 0 or OBS_CHUNK <= 0 or OBS_SAMPLES % OBS_CHUNK:
    raise ValueError('OBS_SAMPLES debe ser positivo y divisible por OBS_CHUNK.')


def gibbs_thermodynamic_limit(T_array, J, h, n_k=GIBBS_K_POINTS):
    """Solución analítica Gibbs del TFIM: energía y correlaciones locales (N -> infinito)."""
    if J <= 0 or h <= J:
        raise ValueError('La solución implementada requiere J>0 y h>J.')
    k = np.linspace(0.0, np.pi, n_k)
    cos_k, sin_k = np.cos(k), np.sin(k)
    q = np.sqrt(J**2 + h**2 - 2.0 * J * h * cos_k)

    def integrate(values):
        return np.trapezoid(values, k) / np.pi

    values = {'T': [], 'energy': [], 'sx': [], 'szz': [], 'sxx': []}
    for temperature in np.asarray(T_array, dtype=float):
        if not np.isfinite(temperature) or temperature <= 0:
            raise ValueError(f'T Gibbs no válida: {temperature}.')
        thermal = np.tanh(q / (2.0 * temperature))
        energy = -integrate(q * thermal)
        sx = integrate((h - J * cos_k) / q * thermal)

        def fermion_G(r):
            numerator = (np.cos(r * k) * (J * cos_k - h)
                         - J * np.sin(r * k) * sin_k)
            return integrate(numerator / q * thermal)

        szz = fermion_G(-1)
        sxx = fermion_G(0)**2 - fermion_G(1) * fermion_G(-1)
        values['T'].append(temperature)
        values['energy'].append(energy)
        values['sx'].append(sx)
        values['szz'].append(szz)
        values['sxx'].append(sxx)

    for key in values:
        values[key] = np.asarray(values[key])
    if not np.allclose(values['energy'], -h * values['sx'] - J * values['szz'], atol=1e-5):
        raise RuntimeError('La energía y los observables Gibbs no satisfacen la identidad del Hamiltoniano.')
    return values

observable_data = {}
common_couplings = None

for N_obs, chi_obs in zip(N_LIST, CHI_LIST):
    data_dir = PROJECT_ROOT / 'data' / '0' / f'mps_N{N_obs}'
    results_file = data_dir / f'results_mps_N{N_obs}_vs_T.json'
    if not results_file.is_file():
        raise FileNotFoundError(f'No hay resultados MPS para N={N_obs}: {results_file}')
    with results_file.open() as f:
        data_n = json.load(f)
    chi_data_n = data_n.get('by_chi', {}).get(str(chi_obs))
    if chi_data_n is None:
        raise KeyError(f'chi={chi_obs} no aparece en los resultados de N={N_obs}.')

    usable = []
    for idx, temperature in enumerate(data_n.get('T', [])):
        if temperature is None or not np.isfinite(temperature) or temperature < OBS_T_MIN:
            continue
        if idx >= len(chi_data_n.get('energy', [])) or chi_data_n['energy'][idx] is None:
            continue
        params_file = data_dir / 'params' / f'params_{idx:04d}_chi{chi_obs}.msgpack'
        if not params_file.is_file():
            print(f'N={N_obs}, chi={chi_obs}, idx={idx}: falta checkpoint; se omite T={temperature:g}.')
            continue
        hx = chi_data_n['h_x'][idx]
        jzz = chi_data_n['J_ZZ'][idx]
        if hx is None or jzz is None:
            raise ValueError(f'Faltan h_x/J_ZZ en N={N_obs}, chi={chi_obs}, idx={idx}.')
        usable.append((idx, float(temperature), float(hx), float(jzz), params_file))

    if not usable:
        raise ValueError(f'No hay temperaturas con parámetros para N={N_obs}, chi={chi_obs}.')
    couplings = (usable[0][2], usable[0][3])
    if any(not np.allclose((row[2], row[3]), couplings) for row in usable):
        raise ValueError(f'h_x/J_ZZ cambian con T para N={N_obs}, chi={chi_obs}.')
    if common_couplings is None:
        common_couplings = couplings
    elif not np.allclose(couplings, common_couplings):
        raise ValueError('Todos los tamaños deben usar el mismo h_x y J_ZZ para compartir Gibbs.')

    h_obs, jzz_obs = couplings
    model_obs = build_model(N_obs, chi_obs, d=MODEL_EMBEDDING,
                            heads=MODEL_HEADS, blocks=MODEL_BLOCKS, mps_init=0.05)
    sampler_obs = make_sampler(model_obs, N_obs, OBS_SAMPLES)
    estimate_obs = make_mps_local_observables(model_obs, N_obs, chi_obs)
    rows = {'T': [], 'energy': [], 'sx': [], 'szz': [], 'sxx': []}

    for idx, temperature, _, _, params_file in usable:
        params_obs = load_params(str(params_file), model_obs, N_obs)
        samples_obs = sampler_obs(params_obs, jax.random.PRNGKey(OBS_SEED + 10000 * N_obs + idx))
        batches = {'sx': [], 'szz': [], 'sxx': []}
        for start in range(0, OBS_SAMPLES, OBS_CHUNK):
            sample_batch = samples_obs[start:start + OBS_CHUNK]
            sx_batch, szz_batch, sxx_batch = estimate_obs(params_obs, sample_batch)
            batches['sx'].append(np.asarray(sx_batch))
            batches['szz'].append(np.asarray(szz_batch))
            batches['sxx'].append(np.asarray(sxx_batch))
        means = {key: float(np.concatenate(value).mean()) for key, value in batches.items()}
        energy_density = h_obs * means['sx'] + jzz_obs * means['szz']
        rows['T'].append(temperature)
        rows['energy'].append(energy_density)
        for key in ('sx', 'szz', 'sxx'):
            rows[key].append(means[key])

    observable_data[(N_obs, chi_obs)] = {
        key: np.asarray(value) for key, value in rows.items()
    }
    print(f'N={N_obs}, chi={chi_obs}: {len(rows["T"])} temperaturas muestreadas.')

all_temperatures = np.concatenate([row['T'] for row in observable_data.values()])
T_gibbs = np.linspace(max(OBS_T_MIN, all_temperatures.min()), all_temperatures.max(), 300)
gibbs_observables = gibbs_thermodynamic_limit(
    T_gibbs, J=abs(common_couplings[1]), h=abs(common_couplings[0])
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
observable_specs = [
    ('sx', r'$\langle\sigma^x\rangle$'),
    ('szz', r'$\langle\sigma^z_i\sigma^z_{i+1}\rangle$'),
    ('sxx', r'$\langle\sigma^x_i\sigma^x_{i+1}\rangle$'),
]
N_cmap = plt.get_cmap('turbo', len(observable_data))
for ax, (key, ylabel) in zip(axes, observable_specs):
    order = np.argsort(gibbs_observables['energy'])
    ax.plot(gibbs_observables['energy'][order], gibbs_observables[key][order],
            color='black', linewidth=2.2, label='Gibbs analítico ($N\\to\\infty$)')
    for color_idx, ((N_obs, chi_obs), values) in enumerate(observable_data.items()):
        ax.scatter(values['energy'], values[key], s=22, alpha=0.8,
                   color=N_cmap(color_idx), label=f'MPS N={N_obs}, chi={chi_obs}')
    ax.set_xlabel(r'$\langle H\rangle/N$')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle('Observables locales frente a energía por sitio')
plt.tight_layout()
plt.show()